In [17]:
import networkx as nx
import numpy as np

## a mudar:

- alphaL,alphaR, gamma, beta fixo
- escolha de influencers só tem a de maior número de arestas (e particularmente 1 de cada cor)
- ele não salva por t

In [ ]:
class GraphModel:

    def __init__(self, num_nodes, num_words, tau):

        self.num_nodes = num_nodes
        self.num_words = num_words
        self.tau = tau

        # alphaL and alphaR parameters (fixed for now)
        self.alphaL = 0.5
        self.alphaR = 0.5

        # gamma and beta parameters (fixed for now)
        self.gamma = 0.2
        self.beta = 0.3

        # word dictionary
        random_vector1 = np.random.rand(num_words)
        self.dictionary = []
        for i in range(num_words):
            self.dictionary.append("blue" if random_vector1[i] > 0.5 else "green")

        # word count by color
        self.num_green_words = self.dictionary.count("green")
        self.num_blue_words = self.dictionary.count("blue")

        # graph initialization
        self.G = nx.Graph()
        random_vector2 = np.random.rand(num_nodes)

        for i in range(num_nodes):
            self.G.add_node(
                i,
                color="green" if random_vector2[i] > 0.5 else "blue"
            )

        # probability vector for each node
        for i in range(num_nodes):

            if self.G.nodes[i]['color'] == "green":
                probs = np.random.dirichlet(
                    self.alphaL * np.ones(self.num_green_words)
                )
            else:
                probs = np.random.dirichlet(
                    self.alphaR * np.ones(self.num_blue_words)
                )

            prob_vector = np.zeros(num_words)
            idx = 0

            for j in range(num_words):
                if self.dictionary[j] == self.G.nodes[i]['color']:
                    prob_vector[j] = probs[idx]
                    idx += 1

            self.G.nodes[i]['prob'] = prob_vector

        # initial edges
        for i in range(num_nodes):
            for j in range(i + 1, num_nodes):
                self.update_edge(i, j)


    def run_cycles(self, num_cycles):
        for _ in range(num_cycles):
            self.step_1()
            self.step_2()
            self.step_3()


    def step_1(self):
        self.influencer_nodes = []
        # For instance, the influencer selection will be the two nodes with the highest degree (one of each color)

        max_degrees = [(-1, -1), (-1,-1)] # (degree, node_index) for green and blue
        for i in range(self.num_nodes):
            degree, color = self.G.degree(i), self.G.nodes[i]['color']
            if color == "green" and degree > max_degrees[0][0]:
                max_degrees[0] = (degree, i)
            elif color == "blue" and degree > max_degrees[1][0]:
                max_degrees[1] = (degree, i)

        self.influencer_nodes = [max_degrees[0][1], max_degrees[1][1]]

        # Update influencer profiles (not specified in detail)


    def step_2(self):

        for i in range(self.num_nodes):
            if i not in self.influencer_nodes:

                Nu = []  # N(u): influencers connected to node u
                for j in self.influencer_nodes:
                    if self.G.has_edge(i, j):
                        Nu.append(j)

                if len(Nu) > 0:
                    profile_sum = np.zeros(self.num_words)
                    for k in Nu:
                        profile_sum += self.G.nodes[k]['prob'] / len(Nu)
                else:
                    profile_sum = self.G.nodes[i]['prob']

                # The new user profile is a convex combination of the current profile and the average influencer profile
                self.G.nodes[i]['prob'] = (
                    (1 - self.gamma) * self.G.nodes[i]['prob'] + self.gamma * profile_sum
                )


    def step_3(self):

        for i in range(self.num_nodes):
            for j in range(i + 1, self.num_nodes):
                self.update_edge(i, j)

    
    def update_edge(self, node1, node2):

        v1 = self.G.nodes[node1]['prob']
        v2 = self.G.nodes[node2]['prob']

        dot_product = np.dot(v1, v2)

        if dot_product > self.tau:
            self.G.add_edge(node1, node2)
        else:
            if self.G.has_edge(node1, node2):
                self.G.remove_edge(node1, node2)


    def cross_group_conection(self): 
        #a measure of bubble burst by counting the fraction of cross-group edges
        total_cross_group_edges_possible = self.num_blue_words * self.num_green_words
        cross_group_edges = 0

        for node1 in range(self.num_nodes):
            for node2 in range(node1 + 1, self.num_nodes):
                color1 = self.G.nodes[node1]['color']
                color2 = self.G.nodes[node2]['color']

                if color1 != color2 and self.G.has_edge(node1, node2):
                        cross_group_edges += 1

        return cross_group_edges / total_cross_group_edges_possible